In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

load_dotenv(r"C:\Users\mauri\Documents\kronos-trading-system\.env", override=True)
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

print("KRONOS v2.0 — Sistema de Señal Diaria")
print("="*50)
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"NewsAPI: {'✓' if NEWS_API_KEY else '✗'}")

KRONOS v2.0 — Sistema de Señal Diaria
Fecha: 2026-04-11 06:12
NewsAPI: ✓


In [2]:
# ── PASO 1: Descargar datos frescos ────────────────────────────
print("Paso 1: Descargando datos frescos...")
eurusd = yf.download("EURUSD=X", period="300d", interval="1d", progress=False)
eurusd.columns = ['Close', 'High', 'Low', 'Open', 'Volume']
eurusd = eurusd.dropna()
print(f"  {len(eurusd)} velas descargadas hasta {eurusd.index[-1].date()}")

# ── PASO 2: Calcular features ──────────────────────────────────
print("\nPaso 2: Calculando features...")

df = eurusd.copy()
df['returns'] = np.log(df['Close'] / df['Close'].shift(1))
df['volatility_20'] = df['returns'].rolling(20).std() * np.sqrt(252)
df['volatility_60'] = df['returns'].rolling(60).std() * np.sqrt(252)

# RSI
delta = df['Close'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
df['rsi_14'] = 100 - (100 / (1 + gain / loss))

# ATR
hl = df['High'] - df['Low']
hc = (df['High'] - df['Close'].shift(1)).abs()
lc = (df['Low'] - df['Close'].shift(1)).abs()
df['atr_14'] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(14).mean()

# Bollinger
df['bb_mid'] = df['Close'].rolling(20).mean()
df['bb_std'] = df['Close'].rolling(20).std()
df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_mid']
df['bb_position'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

# Medias y MACD
df['sma_20'] = df['Close'].rolling(20).mean()
df['sma_50'] = df['Close'].rolling(50).mean()
df['sma_200'] = df['Close'].rolling(200).mean()
df['ema_12'] = df['Close'].ewm(span=12).mean()
df['ema_26'] = df['Close'].ewm(span=26).mean()
df['macd'] = df['ema_12'] - df['ema_26']
df['macd_signal'] = df['macd'].ewm(span=9).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

# Momentum y tendencia
df['momentum_5'] = df['Close'].pct_change(5)
df['momentum_10'] = df['Close'].pct_change(10)
df['momentum_20'] = df['Close'].pct_change(20)
df['trend_20_50'] = df['sma_20'] / df['sma_50'] - 1
df['trend_50_200'] = df['sma_50'] / df['sma_200'] - 1

# Retornos rezagados
df['return_lag_1'] = df['returns'].shift(1)
df['return_lag_2'] = df['returns'].shift(2)
df['return_lag_3'] = df['returns'].shift(3)

df = df.dropna()
print(f"  Features calculados. Última vela: {df.index[-1].date()}")
print(f"  Precio actual EUR/USD: {df['Close'].iloc[-1]:.4f}")

Paso 1: Descargando datos frescos...
  296 velas descargadas hasta 2026-04-10

Paso 2: Calculando features...
  Features calculados. Última vela: 2026-04-10
  Precio actual EUR/USD: 1.1729


In [3]:
# ── PASO 3: Detectar régimen con HMM ──────────────────────────
print("Paso 3: Detectando régimen de mercado...")

with open("../data/processed/hmm_model.pkl", "rb") as f:
    hmm_model = pickle.load(f)
with open("../data/processed/hmm_scaler.pkl", "rb") as f:
    hmm_scaler = pickle.load(f)

features_hmm = df[['returns', 'volatility_20', 'atr_14', 'rsi_14']].copy()
X_hmm = hmm_scaler.transform(features_hmm)
regimenes = hmm_model.predict(X_hmm)
regimen_actual = int(regimenes[-1])
nombres_regimen = {0: 'Tranquilo', 1: 'Normal', 2: 'Turbulento'}

print(f"  Régimen actual: {regimen_actual} — {nombres_regimen[regimen_actual]}")
print(f"  Volatilidad 20d: {df['volatility_20'].iloc[-1]:.4f}")
print(f"  RSI 14: {df['rsi_14'].iloc[-1]:.2f}")

# ── PASO 4: Sentimiento con FinBERT ───────────────────────────
print("\nPaso 4: Analizando sentimiento...")

from transformers import pipeline
finbert = pipeline("text-classification", 
                   model="ProsusAI/finbert", 
                   return_all_scores=True)

def obtener_noticias(query, dias=3):
    fecha = (datetime.now() - timedelta(days=dias)).strftime('%Y-%m-%d')
    url = "https://newsapi.org/v2/everything"
    params = {'q': query, 'from': fecha, 'language': 'en',
              'sortBy': 'publishedAt', 'pageSize': 15, 'apiKey': NEWS_API_KEY}
    r = requests.get(url, params=params)
    data = r.json()
    if data['status'] == 'ok':
        return [a['title'] for a in data['articles'] if a['title']]
    return []

queries = ["EUR USD forex", "ECB Federal Reserve", "eurozone economy"]
titulares = []
for q in queries:
    titulares.extend(obtener_noticias(q))

scores = []
for t in titulares:
    try:
        r = finbert(t[:512])
        sd = {x['label']: x['score'] for x in r}
        scores.append(sd.get('positive', 0) - sd.get('negative', 0))
    except:
        continue

sentiment_score = round(float(np.mean(scores)), 4) if scores else 0.0
sentiment_uncertainty = round(float(np.std(scores)), 4) if scores else 0.5

print(f"  Titulares procesados: {len(titulares)}")
print(f"  Sentiment score: {sentiment_score}")
print(f"  Interpretación: {'Bullish' if sentiment_score > 0.1 else 'Bearish' if sentiment_score < -0.1 else 'Neutral'}")

Paso 3: Detectando régimen de mercado...
  Régimen actual: 1 — Normal
  Volatilidad 20d: 0.0908
  RSI 14: 62.16

Paso 4: Analizando sentimiento...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Titulares procesados: 8
  Sentiment score: -0.5599
  Interpretación: Bearish


In [6]:
# ── PASO 5: Fusion Layer — señal final ────────────────────────
print("Paso 5: Generando señal del Fusion Layer...")

with open("../data/processed/fusion_model_v2.pkl", "rb") as f:
    modelo_fusion = pickle.load(f)

features_v2 = [
    'trend_20_50', 'trend_50_200', 'momentum_20',
    'momentum_10', 'momentum_5', 'volatility_20',
    'volatility_60', 'atr_14', 'rsi_14',
    'macd', 'macd_hist', 'bb_width', 'bb_position',
    'return_lag_1', 'return_lag_2', 'return_lag_3'
]

# Agregar régimen y sentimiento
df['regime'] = regimenes
df['sentiment_score'] = sentiment_score

features_v2_completo = [
    'trend_20_50', 'trend_50_200', 'momentum_20',
    'momentum_10', 'momentum_5', 'volatility_20',
    'volatility_60', 'regime', 'atr_14', 'rsi_14',
    'macd', 'macd_hist', 'bb_width', 'bb_position',
    'sentiment_score', 'return_lag_1', 'return_lag_2', 'return_lag_3'
]

ultimo = df.iloc[-1]
obs = np.array([[ultimo[f] for f in features_v2_completo]])
confianza = modelo_fusion.predict_proba(obs)[0][1]

# ── PASO 6: Aplicar threshold dinámico ────────────────────────
thresholds = {0: None, 1: 0.65, 2: 0.55}
threshold_actual = thresholds.get(regimen_actual)

if threshold_actual is None:
    señal = "HOLD"
    razon = "Régimen Tranquilo — sistema inactivo"
elif confianza > threshold_actual:
    señal = "BUY"
    razon = f"Confianza {confianza:.3f} > {threshold_actual} (Long EUR/USD)"
elif confianza < (1 - threshold_actual):
    señal = "SELL"
    razon = f"Confianza {confianza:.3f} < {1-threshold_actual:.3f} (Short EUR/USD)"
else:
    señal = "HOLD"
    razon = f"Confianza {confianza:.3f} insuficiente (threshold {threshold_actual})"

# ── REPORTE FINAL ──────────────────────────────────────────────
print("\n" + "="*55)
print("   KRONOS v2.0 — SEÑAL DIARIA")
print("="*55)
print(f"   Fecha:       {datetime.now().strftime('%Y-%m-%d')}")
print(f"   EUR/USD:     {ultimo['Close']:.4f}")
print(f"   Régimen:     {regimen_actual} — {nombres_regimen[regimen_actual]}")
print(f"   Volatilidad: {ultimo['volatility_20']:.4f} ({ultimo['volatility_20']*100:.1f}%)")
print(f"   RSI:         {ultimo['rsi_14']:.1f}")
print(f"   Sentimiento: {sentiment_score} ({'Bearish' if sentiment_score < -0.1 else 'Bullish' if sentiment_score > 0.1 else 'Neutral'})")
print(f"   Confianza:   {confianza:.4f}")
print(f"   Threshold:   {threshold_actual}")
print("-"*55)
print(f"   SEÑAL:       *** {señal} ***")
print(f"   Razón:       {razon}")
print("="*55)

# Guardar en log
registro = {
    'fecha': datetime.now().strftime('%Y-%m-%d'),
    'precio': float(ultimo['Close']),
    'regimen': regimen_actual,
    'regimen_nombre': nombres_regimen[regimen_actual],
    'volatilidad': float(ultimo['volatility_20']),
    'rsi': float(ultimo['rsi_14']),
    'sentiment_score': sentiment_score,
    'confianza': float(confianza),
    'threshold': threshold_actual,
    'señal': señal,
    'razon': razon
}

log_path = "../data/processed/paper_trading_log.csv"
df_log = pd.DataFrame([registro])
try:
    existing = pd.read_csv(log_path)
    df_log = pd.concat([existing, df_log], ignore_index=True)
except FileNotFoundError:
    pass

df_log.to_csv(log_path, index=False)
print(f"\n   Señal guardada en paper_trading_log.csv")

Paso 5: Generando señal del Fusion Layer...

   KRONOS v2.0 — SEÑAL DIARIA
   Fecha:       2026-04-11
   EUR/USD:     1.1729
   Régimen:     1 — Normal
   Volatilidad: 0.0908 (9.1%)
   RSI:         62.2
   Sentimiento: -0.5599 (Bearish)
   Confianza:   0.5624
   Threshold:   0.65
-------------------------------------------------------
   SEÑAL:       *** HOLD ***
   Razón:       Confianza 0.562 insuficiente (threshold 0.65)

   Señal guardada en paper_trading_log.csv
